# Notebook 3 — Model Evaluation
**Port Intelligence BI Training Program**

This notebook evaluates the three production models trained in Phase 2:

| Model | File | Task |
|-------|------|------|
| Waiting-time ensemble | `models/waiting_time_ensemble.pkl` | Regression — predict `waiting_anchor_hours` |
| Berth occupancy | `models/berth_occupancy.pkl` | Multi-class classification — occupancy tier |
| Congestion risk | `models/congestion_risk.pkl` | Binary classification — high-congestion flag |

You will reproduce the evaluation split, compute standard metrics, plot diagnostics,
and run a SHAP explainability analysis.

---

## 1  Setup — Load Libraries and Models
All three models are stored as joblib-serialised dictionaries.

In [ ]:
import sys
sys.path.insert(0, '../..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
from sklearn.metrics import (
    mean_absolute_error, r2_score,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve,
)

from features import build_features, ALL_FEATURES

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.05)
plt.rcParams['figure.dpi'] = 110

# ── Load models ───────────────────────────────────────────────────────────────
wt_bundle  = joblib.load('../../models/waiting_time_ensemble.pkl')
bo_bundle  = joblib.load('../../models/berth_occupancy.pkl')
cr_bundle  = joblib.load('../../models/congestion_risk.pkl')

print('=== Waiting-Time Ensemble ===')
print('Keys:', list(wt_bundle.keys()))
print('Metrics:', wt_bundle.get('metrics', {}))

print('\n=== Berth Occupancy Classifier ===')
print('Keys:', list(bo_bundle.keys()))
print('Metrics:', bo_bundle.get('metrics', {}))
print('Label names:', bo_bundle.get('label_names', []))

print('\n=== Congestion Risk Classifier ===')
print('Keys:', list(cr_bundle.keys()))
print('Metrics:', cr_bundle.get('metrics', {}))
print('Congestion threshold (TEU):', cr_bundle.get('congestion_threshold'))
print('Decision threshold:', cr_bundle.get('decision_threshold'))

## 2  Reproduce the 80 / 10 / 10 Time-Series Split
Models were trained on a **time-ordered** split with a 28-day buffer between
each segment to prevent data leakage from rolling-window features.

In [ ]:
GAP_DAYS = 28

# Load and engineer features (this matches training exactly)
raw = pd.read_parquet('../../data/port_calls.parquet')
for col in ['eta_planned', 'ata_actual', 'atb', 'etd', 'atd_actual', 'created_date']:
    if col in raw.columns:
        raw[col] = pd.to_datetime(raw[col])

print('Building features (may take ~30 s) ...')
feat_df = build_features(raw)
feat_df = feat_df.sort_values('ata_actual').reset_index(drop=True)

n = len(feat_df)
train_end_idx = int(n * 0.80)
val_end_idx   = int(n * 0.90)

# Add 28-day gap buffers
train_end_date = feat_df['ata_actual'].iloc[train_end_idx]
val_start_date = train_end_date + pd.Timedelta(days=GAP_DAYS)

val_end_date   = feat_df['ata_actual'].iloc[val_end_idx]
test_start_date = val_end_date + pd.Timedelta(days=GAP_DAYS)

train_mask = feat_df['ata_actual'] <= train_end_date
val_mask   = (feat_df['ata_actual'] >= val_start_date) & (feat_df['ata_actual'] <= val_end_date)
test_mask  = feat_df['ata_actual'] >= test_start_date

feat_cols = [f for f in ALL_FEATURES if f in feat_df.columns]

X_train = feat_df.loc[train_mask, feat_cols].values
X_val   = feat_df.loc[val_mask,   feat_cols].values
X_test  = feat_df.loc[test_mask,  feat_cols].values

y_wait_train = feat_df.loc[train_mask, 'waiting_anchor_hours'].values
y_wait_val   = feat_df.loc[val_mask,   'waiting_anchor_hours'].values
y_wait_test  = feat_df.loc[test_mask,  'waiting_anchor_hours'].values

print(f'\nSplit summary (after 28-day gaps):')
print(f'  Train : {X_train.shape[0]:>6,} rows  '
      f'({feat_df.loc[train_mask, "ata_actual"].min().date()} – '
      f'{feat_df.loc[train_mask, "ata_actual"].max().date()})')
print(f'  Val   : {X_val.shape[0]:>6,} rows  '
      f'({feat_df.loc[val_mask, "ata_actual"].min().date()} – '
      f'{feat_df.loc[val_mask, "ata_actual"].max().date()})')
print(f'  Test  : {X_test.shape[0]:>6,} rows  '
      f'({feat_df.loc[test_mask, "ata_actual"].min().date()} – '
      f'{feat_df.loc[test_mask, "ata_actual"].max().date()})')
print(f'  Feature columns used: {len(feat_cols)}')

## 3  Model 1 — Waiting-Time Ensemble Evaluation
The ensemble blends XGBoost and LightGBM predictions using a learned weight.
Metric: Mean Absolute Error (hours) and R² coefficient of determination.

In [ ]:
xgb_reg     = wt_bundle['xgb_reg']
lgb_reg     = wt_bundle['lgb_reg']
ens_weight  = wt_bundle['ensemble_weight']   # weight for XGB; (1-w) for LGB

y_pred_xgb = xgb_reg.predict(X_test)
y_pred_lgb = lgb_reg.predict(X_test)
y_pred_ens = ens_weight * y_pred_xgb + (1 - ens_weight) * y_pred_lgb
y_pred_ens = np.clip(y_pred_ens, 0, None)   # waiting time >= 0

mae = mean_absolute_error(y_wait_test, y_pred_ens)
r2  = r2_score(y_wait_test, y_pred_ens)

print(f'Ensemble weight (XGB): {ens_weight:.3f}')
print(f'Test MAE : {mae:.2f} h')
print(f'Test R²  : {r2:.4f}')
print(f'\nStored metrics from training: {wt_bundle.get("metrics", {})}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: predicted vs actual
clip_val = 60
sample_idx = np.random.default_rng(0).choice(len(y_wait_test), size=min(3000, len(y_wait_test)), replace=False)
y_act_s  = np.clip(y_wait_test[sample_idx], 0, clip_val)
y_pred_s = np.clip(y_pred_ens[sample_idx],  0, clip_val)

axes[0].scatter(y_act_s, y_pred_s, alpha=0.25, s=10, edgecolors='none', color='steelblue')
lims = [0, clip_val]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual waiting time (h)')
axes[0].set_ylabel('Predicted waiting time (h)')
axes[0].set_title(f'Predicted vs Actual (test set, n={len(sample_idx):,})')
axes[0].text(0.05, 0.92, f'MAE={mae:.2f}h  R²={r2:.3f}',
             transform=axes[0].transAxes, fontsize=10,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[0].legend()

# Residual histogram
residuals = y_wait_test - y_pred_ens
axes[1].hist(np.clip(residuals, -30, 30), bins=70,
             color='#2ca02c', edgecolor='white', linewidth=0.3, alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].axvline(residuals.mean(), color='orange', linestyle='-',
                linewidth=1.5, label=f'Mean residual = {residuals.mean():.2f} h')
axes[1].set_xlabel('Residual (actual − predicted, clipped ±30 h)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution — Waiting-Time Ensemble')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4  Model 2 — Berth Occupancy Classification
The berth-occupancy model predicts which occupancy tier a berth will be in
(e.g. Low / Medium / High / Critical). We evaluate with a classification report
and confusion matrix.

In [ ]:
bo_model      = bo_bundle['model']
bo_features   = bo_bundle.get('features', feat_cols)
label_names   = bo_bundle.get('label_names', None)

# Align feature columns to what the model expects
bo_feat_cols = [f for f in bo_features if f in feat_df.columns]
X_test_bo    = feat_df.loc[test_mask, bo_feat_cols].values

# Build berth-occupancy target: fraction of berths occupied each day×port
# (This reproduces the label creation logic from train_models.py)
berth_counts = {'Haifa': 20, 'Ashdod': 15}
feat_df['berth_occupancy_rate'] = feat_df.apply(
    lambda r: r['arrivals_24h'] / berth_counts[r['port_name']], axis=1
).clip(0, 1)

thresholds = [0.3, 0.6, 0.85]
def _occ_class(rate):
    if rate < thresholds[0]: return 0  # Low
    if rate < thresholds[1]: return 1  # Medium
    if rate < thresholds[2]: return 2  # High
    return 3  # Critical

y_occ_test = feat_df.loc[test_mask, 'berth_occupancy_rate'].apply(_occ_class).values

y_pred_bo = bo_model.predict(X_test_bo)

if label_names is None:
    label_names = ['Low', 'Medium', 'High', 'Critical']

print('Classification Report — Berth Occupancy:')
print(classification_report(y_occ_test, y_pred_bo,
                             target_names=label_names, zero_division=0))

# Confusion matrix heatmap
cm = confusion_matrix(y_occ_test, y_pred_bo)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names,
            linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Confusion Matrix — Berth Occupancy Classifier')
plt.tight_layout()
plt.show()

acc = (y_pred_bo == y_occ_test).mean()
print(f'Overall accuracy: {acc:.4f}')

## 5  Model 3 — Congestion Risk Evaluation
This binary classifier flags calls at risk of high congestion.
We evaluate AUC-ROC, plot the ROC curve, and inspect precision/recall
at the operational decision threshold (0.892).

In [ ]:
cr_model     = cr_bundle['model']
cr_features  = cr_bundle.get('features', feat_cols)
cr_threshold = cr_bundle.get('congestion_threshold', 15)
dec_thresh   = cr_bundle.get('decision_threshold', 0.892)

cr_feat_cols = [f for f in cr_features if f in feat_df.columns]
X_test_cr    = feat_df.loc[test_mask, cr_feat_cols].values

# Reconstruct binary label: waiting_anchor_hours > threshold
y_cr_test = (y_wait_test > cr_threshold).astype(int)

# Predict probabilities
y_prob_cr = cr_model.predict_proba(X_test_cr)[:, 1]
y_pred_cr = (y_prob_cr >= dec_thresh).astype(int)

auc = roc_auc_score(y_cr_test, y_prob_cr)
fpr, tpr, roc_thresh = roc_curve(y_cr_test, y_prob_cr)

print(f'AUC-ROC: {auc:.4f}')
print(f'Decision threshold: {dec_thresh}')
print(f'\nAt threshold {dec_thresh}:')
prec, rec, f1, sup = precision_recall_curve.__doc__  # just for reference
from sklearn.metrics import precision_score, recall_score, f1_score
print(f'  Precision : {precision_score(y_cr_test, y_pred_cr, zero_division=0):.4f}')
print(f'  Recall    : {recall_score(y_cr_test, y_pred_cr, zero_division=0):.4f}')
print(f'  F1        : {f1_score(y_cr_test, y_pred_cr, zero_division=0):.4f}')
print(f'  Prevalence: {y_cr_test.mean():.4f}  ({y_cr_test.sum():,} / {len(y_cr_test):,})')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curve
axes[0].plot(fpr, tpr, color='#1f77b4', linewidth=2,
             label=f'Ensemble (AUC = {auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
# Mark the operating threshold
idx = np.argmin(np.abs(roc_thresh - dec_thresh))
axes[0].scatter(fpr[idx], tpr[idx], s=120, color='red', zorder=5,
                label=f'Threshold = {dec_thresh}')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — Congestion Risk Classifier')
axes[0].legend()

# Score distribution by class
pos = y_prob_cr[y_cr_test == 1]
neg = y_prob_cr[y_cr_test == 0]
axes[1].hist(neg, bins=60, alpha=0.6, color='#1f77b4', label='No congestion', density=True)
axes[1].hist(pos, bins=60, alpha=0.6, color='#d62728', label='High congestion', density=True)
axes[1].axvline(dec_thresh, color='black', linestyle='--', linewidth=1.8,
                label=f'Decision thresh = {dec_thresh}')
axes[1].set_xlabel('Predicted congestion probability')
axes[1].set_ylabel('Density')
axes[1].set_title('Score Distribution by True Class')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6  SHAP Explainability — Waiting-Time XGBoost
SHAP (SHapley Additive exPlanations) attributes each prediction to individual
features. We use a `TreeExplainer` for the XGBoost sub-model.

In [ ]:
try:
    import shap
    shap.initjs()

    # Use 200 randomly sampled test rows
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(len(X_test), size=min(200, len(X_test)), replace=False)
    X_shap = X_test[sample_idx]

    explainer = shap.TreeExplainer(wt_bundle['xgb_reg'])
    shap_values = explainer.shap_values(X_shap)

    print(f'SHAP values shape: {np.array(shap_values).shape}')
    print(f'Feature names passed: {len(feat_cols)}')

    # Summary plot (bar = mean absolute SHAP)
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X_shap,
                      feature_names=feat_cols,
                      plot_type='bar',
                      max_display=20,
                      show=False)
    plt.title('Mean |SHAP| — Top 20 Features (XGBoost waiting-time model)')
    plt.tight_layout()
    plt.show()

    # Beeswarm summary plot
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X_shap,
                      feature_names=feat_cols,
                      max_display=20,
                      show=False)
    plt.title('SHAP Beeswarm — Feature Impact on Waiting-Time Predictions')
    plt.tight_layout()
    plt.show()

    # Print top 10 features by mean |SHAP|
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    shap_df = pd.Series(mean_abs_shap, index=feat_cols).sort_values(ascending=False)
    print('Top 10 features by mean |SHAP|:')
    print(shap_df.head(10).round(4).to_string())

except ImportError:
    print('shap not installed. Run: pip install shap')
    print('Skipping SHAP analysis.')

## 7  Threshold Analysis — Congestion Risk Model
The operating threshold is a business decision. Sweeping it from 0 to 1 reveals
the precision–recall trade-off so operators can choose the right balance.

In [ ]:
thresholds_sweep = np.linspace(0.0, 1.0, 201)
precisions = []
recalls    = []
f1_scores  = []

for t in thresholds_sweep:
    y_t = (y_prob_cr >= t).astype(int)
    precisions.append(precision_score(y_cr_test, y_t, zero_division=0))
    recalls.append(recall_score(y_cr_test, y_t, zero_division=0))
    f1_scores.append(f1_score(y_cr_test, y_t, zero_division=0))

precisions = np.array(precisions)
recalls    = np.array(recalls)
f1_scores  = np.array(f1_scores)

best_f1_idx = np.argmax(f1_scores)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Precision and recall vs threshold
axes[0].plot(thresholds_sweep, precisions, label='Precision', color='#1f77b4', linewidth=2)
axes[0].plot(thresholds_sweep, recalls,    label='Recall',    color='#d62728', linewidth=2)
axes[0].plot(thresholds_sweep, f1_scores,  label='F1',        color='#2ca02c', linewidth=2, linestyle='-.')
axes[0].axvline(dec_thresh, color='black', linestyle='--', linewidth=1.5,
                label=f'Operational thresh = {dec_thresh}')
axes[0].axvline(thresholds_sweep[best_f1_idx], color='purple', linestyle=':',
                linewidth=1.5,
                label=f'Best F1 @ {thresholds_sweep[best_f1_idx]:.3f}')
axes[0].set_xlabel('Decision threshold')
axes[0].set_ylabel('Metric value')
axes[0].set_title('Precision / Recall / F1 vs Threshold')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1.05)

# Precision–Recall curve
pr_precision, pr_recall, pr_thresh = precision_recall_curve(y_cr_test, y_prob_cr)
axes[1].plot(pr_recall, pr_precision, color='#1f77b4', linewidth=2,
             label=f'PR curve (AUC={auc:.4f})')
# Mark operating point
op_idx = np.argmin(np.abs(pr_thresh - dec_thresh)) if len(pr_thresh) > 0 else 0
axes[1].scatter(pr_recall[op_idx], pr_precision[op_idx],
                s=120, color='red', zorder=5,
                label=f'Operational point\nP={pr_precision[op_idx]:.3f}, R={pr_recall[op_idx]:.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision–Recall Curve — Congestion Risk')
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print(f'Best threshold by F1 : {thresholds_sweep[best_f1_idx]:.3f}  '
      f'(F1={f1_scores[best_f1_idx]:.4f})')
print(f'Operational threshold: {dec_thresh}  '
      f'(F1={f1_scores[np.argmin(np.abs(thresholds_sweep - dec_thresh))]:.4f})')
print('\nNote: a higher threshold favours precision (fewer false alarms);')
print('a lower threshold favours recall (catch more true congestion events).')

## 8  Exercises

**Exercise 1** — Compute the MAE separately for Haifa and Ashdod test rows.
Which port is harder to predict?

**Exercise 2** — For the congestion-risk model, find the threshold that achieves
at least 0.90 recall. What is the precision at that threshold?

**Exercise 3** — Plot a calibration curve (reliability diagram) for the congestion
risk model: compare mean predicted probability vs fraction of positives in 10 bins.

In [ ]:
# Exercise 1 — MAE by port
# We need port_name for the test rows

test_ports = feat_df.loc[test_mask, 'port_name'].values

for port in ['Haifa', 'Ashdod']:
    mask_port = test_ports == port
    mae_port = mean_absolute_error(
        y_wait_test[mask_port],
        y_pred_ens[mask_port]   # YOUR CODE HERE — use y_pred_ens
    )
    print(f'{port}: MAE = {mae_port:.2f} h  (n={mask_port.sum():,})')

# Which port has higher MAE?
# YOUR CODE HERE: print a conclusion

In [ ]:
# Exercise 2 — Find threshold where recall >= 0.90

target_recall = 0.90

# Hint: recalls and thresholds_sweep are already computed above
# Find the highest threshold where recall >= target_recall

valid_mask = np.array(recalls) >= ???         # YOUR CODE HERE
if valid_mask.any():
    best_idx = np.where(valid_mask)[0].???()  # YOUR CODE HERE: max() to get highest threshold
    t_found = thresholds_sweep[best_idx]
    p_found = precisions[best_idx]
    r_found = recalls[best_idx]
    print(f'At threshold {t_found:.3f}: Precision={p_found:.4f}, Recall={r_found:.4f}')
else:
    print(f'Cannot achieve recall >= {target_recall} — try a lower threshold.')

In [ ]:
# Exercise 3 — Calibration curve (reliability diagram)

n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
mean_predicted = []
fraction_pos   = []

for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    in_bin = (y_prob_cr >= lo) & (y_prob_cr < hi)
    if in_bin.sum() > 0:
        mean_predicted.append(y_prob_cr[in_bin].mean())
        fraction_pos.append(???[in_bin].mean())  # YOUR CODE HERE: y_cr_test

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Perfect calibration')
ax.plot(mean_predicted, fraction_pos, marker='o', linewidth=2,
        color='#1f77b4', label='Congestion risk model')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration Curve — Congestion Risk')
ax.legend()
plt.tight_layout()
plt.show()